# AutoIOT Paper Sanity Check (MIT ECG)

This notebook runs a **single live end-to-end** AutoIOT-paper baseline pass using the reconstructed pipeline in `flashfusion/baselines/autoiot_paper.py`.

## Prerequisites

- `GROQ_API_KEY` exported
- `TAVILY_API_KEY` exported
- Run from this repository workspace

## What this validates

- Terms -> generated web search queries -> retrieval
- High-level + detailed design
- Per-module code generation + code integration
- Iterative execution feedback with stderr/stdout-aware refinement
- Best-version selection and final output mapping into `RunResult`

In [5]:
# Install exact package versions matching the project's working environment.
import subprocess, sys
pkgs = [
    "langchain==1.2.10",
    "langchain-core==1.2.16",
    "langchain-groq==1.1.2",
    "langchain-community==0.4.1",
    "tavily-python==0.7.22",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Dependencies ready.")


Dependencies ready.


In [6]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root robustly from this notebook location.
cwd = Path.cwd().resolve()
if not (cwd / "flashfusion").exists():
    repo_root = None
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "flashfusion").exists():
            repo_root = candidate
            break
    if repo_root is None:
        raise RuntimeError("Could not locate repository root containing 'flashfusion/'.")
    os.chdir(repo_root)

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from flashfusion.baselines.autoiot_paper import run_autoiot_paper
from flashfusion.config import DEFAULT_MODEL
from flashfusion.pipeline.loader import load_mit_arrythmia
from flashfusion.pipeline.runner import LLMClient, RunResult

print(f"Repo root: {repo_root}")
print(f"Default model: {DEFAULT_MODEL}")


Repo root: /Users/kausar/Documents/ff-context/flash-fusion
Default model: llama-3.3-70b-versatile


In [10]:
# Load API keys from the repo vault (.env at repo root).
# vault.py calls load_dotenv(repo_root/.env, override=False),
# so keys already exported in the shell take precedence.
import vault  # noqa: F401 — side-effect import

missing = [k for k in ("GROQ_API_KEY", "TAVILY_API_KEY") if not os.getenv(k)]
if missing:
    raise EnvironmentError(
        f"Missing API keys: {missing}\n"
        f"Paste your keys into {repo_root / '.env'} and re-run this cell."
    )
print("API keys confirmed:", [k for k in ("GROQ_API_KEY", "TAVILY_API_KEY")])


API keys confirmed: ['GROQ_API_KEY', 'TAVILY_API_KEY']


In [8]:
# --- Run configuration (single-query sanity check) ---
MODEL_NAME = DEFAULT_MODEL
ECG_PATH = repo_root / "data/AutoIOT_dataset/ECG.0/MIT_arrythmia_v1.txt"

QUERY = (
    "For each ECG record_id in this dataset, detect R-peaks from MLII and report "
    "a detection-accuracy style summary using available annotations."
)

# Keep first pass fast; switch to False for full-data run.
FAST_SANITY = True
MAX_ROWS = 120_000
TARGET_RECORD_IDS = {101, 106, 208}

if not ECG_PATH.exists():
    raise FileNotFoundError(f"MIT ECG file not found: {ECG_PATH}")

print(f"ECG path: {ECG_PATH}")
print(f"Model:    {MODEL_NAME}")
print(f"Query:    {QUERY}")
print(f"FAST_SANITY={FAST_SANITY}, MAX_ROWS={MAX_ROWS}, TARGET_RECORD_IDS={sorted(TARGET_RECORD_IDS)}")


ECG path: /Users/kausar/Documents/ff-context/flash-fusion/data/AutoIOT_dataset/ECG.0/MIT_arrythmia_v1.txt
Model:    llama-3.3-70b-versatile
Query:    For each ECG record_id in this dataset, detect R-peaks from MLII and report a detection-accuracy style summary using available annotations.
FAST_SANITY=True, MAX_ROWS=120000, TARGET_RECORD_IDS=[101, 106, 208]


In [9]:
df = load_mit_arrythmia(str(ECG_PATH))
print(f"Loaded MIT ECG rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

if FAST_SANITY:
    df = df[df["record_id"].isin(TARGET_RECORD_IDS)].head(MAX_ROWS).copy()

record_ids = sorted(df["record_id"].dropna().astype(int).unique().tolist())
print(f"Sanity dataframe rows: {len(df):,}")
print(f"Record IDs present: {record_ids}")
display(df.head(3))


Loaded MIT ECG rows: 26,000,000
Columns: ['sample_idx', 'time_s', 'MLII', 'V1', 'record_id', 'annotation']
Sanity dataframe rows: 120,000
Record IDs present: [101]


,sample_idx,time_s,MLII,V1,record_id,annotation
0,0,0.000000,-0.345,-0.16,101,
1,1,0.002778,-0.345,-0.16,101,
2,2,0.005556,-0.345,-0.16,101,


In [11]:
client = LLMClient(model_name=MODEL_NAME, api_key=os.environ["GROQ_API_KEY"])
result = RunResult(baseline="AUTOIOT_PAPER", model=MODEL_NAME, query=QUERY)

result = run_autoiot_paper(QUERY, df, client, result)

print("Run complete.")
print(f"executed={result.executed}, rejected={result.rejected}, tries={result.agent_tries}")
print(f"stages_run={result.stages_run}")


Run complete.
executed=True, rejected=False, tries=15
stages_run=['autoiot_terms', 'autoiot_search_queries', 'autoiot_retrieval', 'autoiot_design_high', 'autoiot_design_detail', 'autoiot_module_gen', 'autoiot_code_integration', 'autoiot_agent_loop', 'autoiot_select']


In [12]:
print("Answer preview:")
print((result.answer or "")[:1000])
print("\nFinal code preview:")
print((result.final_code or "")[:1000])


Answer preview:
```python
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, butter, lfilter

def detect_r_peaks(ecg_data):
    """
    Detect R-peaks from ECG data using a simple peak detection algorithm.

    Parameters:
    ecg_data (pd.DataFrame): ECG data with 'sample_idx', 'time_s', 'MLII', 'V1', 'record_id', and 'annotation' columns.

    Returns:
    r_peaks (list): List of detected R-peak indices.
    """
    # Filter the ECG data to only include the MLII lead
    mlii_data = ecg_data['MLII']

    # Apply a 5 Hz low-pass filter to the MLII data to enhance the QRS complex
    nyq = 0.5 * 360  # Assuming a sampling rate of 360 Hz
    low_cutoff = 5 / nyq
    b, a = butter(5, low_cutoff, btype='low')
    filtered_data = lfilter(b, a, mlii_data)

    # Square the filtered data to further enhance the QRS complex
    squared_data = np.square(filtered_data)

    # Detect peaks in the squared data using a dynamic threshold
    peaks, _ = find_peaks(squared_dat

In [13]:
expected_stages = [
    "autoiot_terms",
    "autoiot_search_queries",
    "autoiot_retrieval",
    "autoiot_design_high",
    "autoiot_design_detail",
    "autoiot_module_gen",
    "autoiot_code_integration",
    "autoiot_agent_loop",
    "autoiot_select",
]

missing_stages = [s for s in expected_stages if s not in result.stages_run]
if missing_stages:
    raise AssertionError(f"Missing expected stages: {missing_stages}")

print("All expected AutoIOT stages are present.")


All expected AutoIOT stages are present.


In [14]:
if not result.execution_attempts:
    raise AssertionError("execution_attempts is empty; expected iteration records.")

best_record = result.execution_attempts[-1]
required_record_keys = [
    "search_queries",
    "retrieval_provenance",
    "module_codes",
    "execution_feedback",
    "corrected_code",
]
missing_keys = [k for k in required_record_keys if k not in best_record]
if missing_keys:
    raise AssertionError(f"Missing expected iteration-record keys: {missing_keys}")

feedback = best_record.get("execution_feedback", {}) or {}
feedback_missing = [k for k in ["status", "stdout", "stderr"] if k not in feedback]
if feedback_missing:
    raise AssertionError(f"Missing execution feedback fields: {feedback_missing}")

provenance = best_record.get("retrieval_provenance", []) or []
generated_queries = [p.get("generated_query") for p in provenance if isinstance(p, dict)]
generated_queries = [q for q in generated_queries if q]

print(f"Iteration records: {len(result.execution_attempts)}")
print(f"Module code segments in record: {len(best_record.get('module_codes', []))}")
print(f"Retrieval provenance entries: {len(provenance)}")
print(f"Generated queries captured: {len(generated_queries)}")
print(f"Feedback status: {feedback.get('status')}")
print("Sample generated queries:")
print(generated_queries[:5])


Iteration records: 5
Module code segments in record: 17
Retrieval provenance entries: 18
Generated queries captured: 18
Feedback status: success
Sample generated queries:
['ECG R-peak detection algorithms', 'ECG MLII lead signal processing', 'ECG annotation based R-peak detection accuracy metrics', 'R-peaks detection algorithms', 'ECG R-peaks annotation guidelines']


## Parity Note vs Legacy AutoIOT Notebook

Conceptual mapping from legacy notebook flow in `autoiot/notebooks/main_ECG.ipynb` to reconstructed `autoiot_paper.py` stages:

- term determination -> `autoiot_terms`
- concept search URL generation -> `autoiot_search_queries` + `autoiot_retrieval`
- high-level design -> `autoiot_design_high`
- detailed design -> `autoiot_design_detail`
- per-step code generation -> `autoiot_module_gen`
- final code integration -> `autoiot_code_integration`
- iterative execution/refinement -> `autoiot_agent_loop`
- best-version decision -> `autoiot_select`